
# 🧱 Databricks Course — Day 3 Notes
> **Focus:** Unity Catalog — Architecture, Object Model, Setup & Cloud Storage Access (Azure)

---

## 1. 🕰️ Before Unity Catalog — The Legacy Problem

### What existed before:
- **DBFS (Databricks File System)** — a file system abstraction, workspace-scoped, no fine-grained access control
- **Hive Metastore** — stored metadata (Tables, Views, Functions) but was **per-workspace only**

### The Problems with Legacy:
- Each workspace had its **own isolated Hive Metastore** — no sharing across workspaces
- No centralized governance — different teams, different permissions, no single source of truth
- No data lineage — couldn't track where data came from or who accessed it
- No column/row level security out of the box
- Compliance (GDPR, HIPAA) was painful to enforce

```
Legacy World:
Workspace A  →  Hive Metastore A  →  ADLS
Workspace B  →  Hive Metastore B  →  ADLS   ← No sharing, no central control
Workspace C  →  Hive Metastore C  →  ADLS
```

---

## 2. ✅ What is Unity Catalog & Key Features

> **Unity Catalog = Centralized governance layer for all data and AI assets across ALL workspaces**

```
Recommended World (from course slides):

Users & Apps → Compute (Apache Spark)
                      ↕  Read / Write
              [ Unity Catalog ]           ← single governance layer
              Volume | Table | View | Function
                      ↕
              Data Storage (ADLS)
```

### Key Features:

| Feature | What it means |
|---|---|
| **Centralized Metastore** | One metastore shared across multiple workspaces |
| **3-Level Namespace** | `catalog.schema.table` — clean, organized hierarchy |
| **Fine-grained Access Control** | Grant/Revoke at catalog, schema, table, column, or row level |
| **Data Lineage** | Automatically tracks how data flows between tables |
| **Audit Logs** | Who accessed what, when — critical for compliance |
| **Delta Sharing** | Share data with external orgs without copying it |
| **Works across clouds** | Azure, AWS, GCP — same governance model |

**⚠️ Production Tip:** Unity Catalog replaces both DBFS and Hive Metastore in modern Databricks setups. Any new project on Azure Databricks should use Unity Catalog from day one.

---

## 3. 🏗️ Unity Catalog Object Model

This is the full hierarchy — think of it as nested containers:

```
┌─────────────────────────────────────────────────────────┐
│                      METASTORE                          │
│         (Top-level container — 1 per region)            │
│         Paired with a default ADLS storage account      │
│                                                         │
│  ┌──────────────────────────────────────────────────┐   │
│  │                   CATALOG                        │   │
│  │   (Logical grouping — e.g. by team/project/env)  │   │
│  │   e.g.  catalog_bronze, catalog_silver, catalog_gold  │
│  │                                                  │   │
│  │  ┌────────────────────────────────────────────┐  │   │
│  │  │              SCHEMA (Database)             │  │   │
│  │  │  Next level container — Schema = Database  │  │   │
│  │  │  Use "schema" not "database" going forward │  │   │
│  │  │                                            │  │   │
│  │  │  ┌──────────┐ ┌───────┐ ┌──────┐ ┌──────┐ │  │   │
│  │  │  │  VOLUME  │ │ TABLE │ │ VIEW │ │ FUNC │ │  │   │
│  │  │  └──────────┘ └───────┘ └──────┘ └──────┘ │  │   │
│  │  └────────────────────────────────────────────┘  │   │
│  └──────────────────────────────────────────────────┘   │
└─────────────────────────────────────────────────────────┘
```

---

### 🔍 Each Object Explained:

#### 📦 Metastore
- **Top-level container** — holds everything
- **Only ONE metastore per region** — shared across all workspaces in that region
- Paired with a default ADLS Gen2 storage account (where managed data lives)
- Think of it as the "root" of your entire data organization

#### 🗂️ Catalog
- **Newly introduced** in Unity Catalog (didn't exist in Hive Metastore)
- Logical grouping inside the metastore
- Use it to separate environments or domains: `dev_catalog`, `prod_catalog`, `finance_catalog`, `marketing_catalog`
- Each catalog can be mapped to a **specific ADLS storage container**

#### 🗃️ Schema (= Database)
- Same as a database — just called "schema" in Unity Catalog terminology
- Groups related tables, views, functions together
- e.g. `prod_catalog.sales.orders`, `prod_catalog.sales.customers`

#### 📁 Volume
- Stores **files** (non-tabular data) — CSV, JSON, images, logs, etc.
- Two types:
  - **Managed Volume** — Unity Catalog manages the storage location (inside default ADLS)
  - **External Volume** — you point it to your own ADLS path

#### 📊 Table
- Stores **structured/tabular data**
- Two types:
  - **Managed Table** — UC manages location + lifecycle. **Always Delta format.** Delete table = data deleted.
  - **External Table** — you control the storage path. Delete table = only metadata deleted, data stays.
- ⚠️ **Important Rule:** All managed tables in Unity Catalog are **Delta tables only**. You CANNOT create a managed table in Parquet, CSV, or JSON. For those formats, you must create an **External Table**.

#### 👁️ View
- Virtual table based on a SQL query
- No data stored — always computed on the fly
- Great for exposing filtered/transformed data without duplicating it
- Use for row/column level security (show only certain rows to certain users)

#### ƒ Function
- Stored reusable SQL or Python logic
- Call it like `SELECT my_catalog.my_schema.my_function(column)` from any query

---

## 4. 🔧 Unity Catalog Setup — Step by Step

### Pre-requisites:
- User must have **Global Administrator** privileges on the Azure tenant
- Azure subscription created **on or after November 2023** → Unity Catalog metastore is created **automatically by default** ✅
- Older subscriptions → need to manually create the metastore

### Setup Steps:

```
Step 1: Login to Databricks Account Console
         → accounts.azuredatabricks.net (not workspace URL)
         → This is the account-level admin portal

Step 2: Create Unity Catalog Metastore
         → Account Console → Data → Create Metastore
         → Give it a name, select region, assign default ADLS storage path

Step 3: Assign Databricks Workspace to the Metastore
         → A workspace can only be assigned to ONE metastore
         → Account Console → Workspaces → Assign to Metastore

Step 4: Configure Cluster to support Unity Catalog
         → Go to Compute → Select your cluster → Edit
         → Access Mode must be: "Single User" or "Shared" (NOT No Isolation Shared)
         → No Isolation Shared does NOT support Unity Catalog
```

### Databricks Account Console — Important Sections:

| Section | What's here |
|---|---|
| **Workspaces** | View/manage all workspaces, assign to metastore |
| **Users & Groups** | Manage who has access at account level |
| **Data (Metastore)** | Create and manage metastores |
| **Settings** | Account-level configs, SSO, SCIM provisioning |

**⚠️ Production Tip:** The Account Console is different from the Workspace UI. Most developers only see the Workspace. Only Account Admins have access to the Account Console. In production, the setup is typically done by a platform/infra team — but you need to understand it to debug access issues.

---

## 5. ☁️ Configure Access to Cloud Storage (Azure)

This is how Spark (compute) actually reads and writes to ADLS (data storage) **through** Unity Catalog.

### The Architecture (from course slides):

```
Users & Apps
     ↓
Compute (Apache Spark)
     ↕  Read / Write
┌─────────────────────────────────────────────────┐
│                Unity Catalog                    │
│                                                 │
│   Metastore                                     │
│        ↓                                        │
│   Storage Credential ←──── Access Connector     │
│   (Authentication)          for Databricks      │
│        +                                        │
│   External Location  ←──── Storage Credential  │
│   (Auth + Container)  +──── ADLS Container      │
└─────────────────────────────────────────────────┘
     ↕                              ↕
  ADLS (default)            ADLS Container
  (managed tables)          (external tables)
```

### The Concept — Why 2 Objects?

You need **2 things** inside Unity Catalog to connect to an ADLS container:

| Object | Purpose |
|---|---|
| **Storage Credential** | Handles **authentication** — proves to Azure "I am allowed to access this storage" |
| **External Location** | Combines Storage Credential + ADLS Container path — grants access to a **specific container** |

You can create **as many External Locations as you need** — one per container, or one per catalog/layer. For example: one for Bronze, one for Silver, one for Gold → each mapped to its own ADLS container.

---

### 🔑 Storage Credential — How it Works on Azure

Azure offers a first-party service called **Access Connector for Databricks** — this is a managed identity that Unity Catalog uses to authenticate to ADLS without any passwords or keys.

### ✅ Full Setup Steps — 5 Steps:

```
Step 1: Create Access Connector for Databricks
         → Azure Portal → Create Resource → "Access Connector for Databricks"
         → This creates a Managed Identity in Azure

Step 2: Create Azure Data Lake Storage (ADLS Gen2)
         → Azure Portal → Create Storage Account
         → Enable "Hierarchical Namespace" (required for ADLS Gen2)
         → Create a container inside it (e.g. "bronze-container")

Step 3: Assign "Storage Blob Data Contributor" Role
         → Go to your ADLS storage account → IAM → Add Role Assignment
         → Role: Storage Blob Data Contributor
         → Assign to: the Access Connector managed identity from Step 1
         → This gives the connector read/write permission to the storage

Step 4: Create Storage Credential in Unity Catalog
         → Databricks Account Console → Unity Catalog → Storage Credentials → Create
         → Paste the Access Connector resource ID
         → This registers the authentication method in UC

Step 5: Create External Location in Unity Catalog
         → Unity Catalog → External Locations → Create
         → Select the Storage Credential from Step 4
         → Provide the ADLS container path (abfss://container@storageaccount.dfs.core.windows.net/)
         → Now Spark can read/write to this container through UC
```

### Visual Flow:

```
Azure AD / Managed Identity
         ↓
Access Connector for Databricks  (Step 1)
         ↓  (assigned Storage Blob Data Contributor)
ADLS Gen2 Container              (Step 2 + 3)
         ↑
Storage Credential in UC         (Step 4)  ← authentication wrapper
         +
External Location in UC          (Step 5)  ← auth + specific path
         ↓
Catalog 1 → maps to → External Location 1 → Container A
Catalog 2 → maps to → External Location 2 → Container B
```

**⚠️ Production Tip:** Never use storage account keys or SAS tokens to connect Databricks to ADLS in production. Always use the Access Connector (Managed Identity) approach — it's more secure, no secrets to rotate, and auditable. This is the Azure-recommended and Databricks-recommended pattern.

**⚠️ Production Tip:** One External Location per environment layer is a clean pattern — `ext_loc_bronze`, `ext_loc_silver`, `ext_loc_gold` each pointing to separate containers. This gives you independent access control per layer.

---

## 📌 Day 3 — Quick Recap

```
Before UC       → Per-workspace Hive Metastore + DBFS — no central governance
Unity Catalog   → Centralized governance, lineage, audit, fine-grained access control

Object Model Hierarchy:
Metastore (1 per region)
  └── Catalog (logical grouping, maps to storage container)
        └── Schema = Database
              ├── Volume     (files — managed or external)
              ├── Table      (structured data — managed=Delta only, external=any format)
              ├── View       (virtual, SQL-based, used for security/abstraction)
              └── Function   (reusable SQL/Python logic)

Setup Order:
Account Console → Create Metastore → Assign Workspace → Configure Cluster (Shared/Single User)

Cloud Storage Access (Azure):
Access Connector → ADLS → Blob Data Contributor Role → Storage Credential → External Location
```

---